<a href="https://colab.research.google.com/github/ANNYM25/python-together/blob/main/08-machine-learning/05_sklearn_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5 · A real scikit-learn pipeline

Up to now we hand-built models. In practice the *unit of work* isn't a lone
model — it's **prepare the data → model → evaluate honestly**, bundled so the
preparation can't accidentally peek at the test set. That bundle is a
`Pipeline`, and it's how professionals keep themselves honest.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## An imbalanced dataset (like real risk data)

80% one class, 20% the other — so we remember lesson 3 about accuracy lying.

In [ ]:
X, y = make_classification(
    n_samples=400, n_features=6, n_informative=4, n_redundant=0,
    weights=[0.8, 0.2], random_state=0,
)
print("class balance:", {0: int((y == 0).sum()), 1: int((y == 1).sum())})

## Split, then build the pipeline

`stratify=y` keeps the same class balance in train and test. The pipeline chains
a **scaler** (puts every feature on a comparable scale) and the **model**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0
)

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500)),
])
pipe.fit(X_train, y_train)

## Judge it with the honest metrics

`classification_report` prints precision / recall / F1 **per class** — exactly
the lesson-3 numbers, computed for you.

In [ ]:
y_hat = pipe.predict(X_test)
print(classification_report(y_test, y_hat, digits=3))

## Cross-validation — a more trustworthy score

A single train/test split can be lucky or unlucky. **k-fold cross-validation**
rotates the test fold `k` times and averages, so the score wobbles less.

In [ ]:
scores = cross_val_score(pipe, X, y, cv=5, scoring="f1")
print("5-fold F1 scores:", scores.round(3))
print("mean F1:", round(scores.mean(), 3), "±", round(scores.std(), 3))

## 🏋️ Your turn

A classifier can output a **probability**, not just a label. Print
`pipe.predict_proba(X_test)` for the first 5 test rows — each row is
`[P(class 0), P(class 1)]`. Then count how many test rows the model is more than
90% sure are class 1.

In [ ]:
probs = pipe.predict_proba(X_test)
print(probs[:5].round(3))
# very_sure = ...   # count rows where probs[:, 1] > 0.9

### ✅ What you learned
- A `Pipeline` bundles preprocessing + model so test data never leaks into fitting.
- `StandardScaler` puts features on a comparable scale (many models need this).
- `classification_report` gives per-class precision/recall/F1.
- **Cross-validation** averages over several splits for a steadier score.
- `predict_proba` exposes confidence, not just a hard label.

**Next:** `06_unsupervised.py` — finding structure with no labels at all.